# Standard Python Logging Module

In [1]:
import sys

REQUIRED = (3, 8)
assert sys.version_info >= REQUIRED, (
    f"Python {REQUIRED[0]}.{REQUIRED[1]}+ required, "
    f"got {sys.version}"
)

In [2]:
import logging

logging.debug("This is a debug message")       #10
logging.info("This is an info message")        #20
logging.warning("This is a warning message")   #30
logging.error("This is an error message")      #40
logging.critical("This is a critical message") #50

ERROR:root:This is an error message
CRITICAL:root:This is a critical message


In [3]:
import logging

logging.basicConfig(level=logging.DEBUG, force=True)

logging.debug("This is a debug message")
logging.info("This is an info message")
logging.warning("This is a warning message")
logging.error("This is an error message")
logging.critical("This is a critical message")

DEBUG:root:This is a debug message
INFO:root:This is an info message
ERROR:root:This is an error message
CRITICAL:root:This is a critical message


### Configure Global Logger

#### Standard Error Stream

In [4]:
import logging

logging.basicConfig(
    format="{asctime} : {name} : {levelname} : {message}",
    style="{",
    datefmt="%Y-%m-%d %H:%M",
    level=logging.DEBUG, 
    force=True,
    )

logging.debug("This is a DEBUG message")

2026-02-16 14:54 : root : DEBUG : This is a DEBUG message


#### File Stream and Excepiton Handling

In [5]:
import logging
import sys

logging.basicConfig(
    filename="project.log",
    encoding="utf-8",
    filemode="a",
    format="{asctime} : {name} : {levelname} : {message}",
    style="{",
    datefmt="%Y-%m-%d %H:%M",
    level=logging.DEBUG, 
    force=True,
    )

title = "Literature Review"
logging.debug(f"{title=}")

articles = 5
researchers = 0

# Catch exceptions
# 1
try:
    workload = articles / researchers
except ZeroDivisionError as e:
    print("ERROR WorkloadError", e, file=sys.stderr)
print("Passed exception and printed error message")

# 2
try:
    workload = articles / researchers
except ZeroDivisionError:
    logging.error("WorkloadError", exc_info=True)
print("Passed traditional exception logging")

# 3
try:
    workload = articles / researchers
except ZeroDivisionError:
    logging.exception("WorkloadError")
print("Passed cleaner logging alternative")

Passed exception and printed error message
Passed traditional exception logging
Passed cleaner logging alternative


ERROR WorkloadError division by zero


### Custom Logger

In [6]:
import logging

logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")

#### Clean Handlers from Root and Custom Loggers

In [7]:
def clean_handlers(logger):
    '''Clear existing handlers before adding new ones'''
    if hasattr(logger, 'handlers') and logger.handlers:
        for handler in logger.handlers[:]:  
            logger.removeHandler(handler)

# Also clean the root logger to remove any basicConfig handlers
def clean_root_logger():
    '''Remove all handlers from root logger'''
    root_logger = logging.getLogger()
    for handler in root_logger.handlers[:]:
        root_logger.removeHandler(handler)

#### Two-Stage Filtering: Logger vs Handler Levels
   - Logger level acts as a "source filter" 
   - Handler level acts as a "destination filter"

In [8]:
import logging

def show_only_warning(record):
    return record.levelname == "WARNING"

# Clean existing handlers
clean_handlers(logger)
clean_root_logger()

# Set up custom logger
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")

# Set up handler
file_handler = logging.FileHandler("project.log", mode="a", encoding="utf-8")
file_handler.setLevel("INFO")
file_handler.setFormatter(logging.Formatter("{levelname} - {message}", style="{"))
file_handler.addFilter(show_only_warning)
logger.addHandler(file_handler)

# Run custom logger
logger.debug("Post DEBUG record")
logger.warning("Post WARNING record")
logger.error("Post ERROR record")

#### Custom Logger Streaming to Console and File

[ LogRecord INFO ]
        |
        v
[ Logger (DEBUG) ]
        |
        v
[ Handler Router ]
     /         \
    v           v
[ FileHandler ] [ StreamHandler ]
   (INFO)          (ERROR)
     |                |
     v                X
  records.log       stderr

In [9]:
import logging


# Clean existing handlers
clean_handlers(logger)
clean_root_logger()

# Set up custom logger first
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")

# Set up CONSOLE handler 
h1 = logging.StreamHandler()
h1.setLevel("DEBUG")
h1.setFormatter(logging.Formatter("{levelname} - {message}", style="{"))
logger.addHandler(h1)

# Set up FILE handler 
h2 = logging.FileHandler("project.log", mode="a", encoding="utf-8")
h2.setLevel("WARNING")
h2.setFormatter(logging.Formatter("{asctime} : {name} : {levelname} : {message}", style="{"))
logger.addHandler(h2)

# Run custom logger
logger.debug("I'm DEBUG record")
logger.warning("I'm WARRNING record")
logger.error("I'm ERROR record")

DEBUG - I'm DEBUG record
WARNING - I'm WARRNING record
ERROR - I'm ERROR record
